# Importing Libraries

In [1]:
import sagemaker
import boto3
from sagemaker.amazon.amazon_estimator import get_image_uri
from sagemaker.session import s3_input, Session

# Create S3 Bucket

In [2]:
bucket_name= 'flightfareprediction'
my_region=boto3.session.Session().region_name  #set the region of the instance
print(my_region)

us-east-1


In [3]:
s3=boto3.resource('s3') 
try:
    if my_region == 'us-east-1':
        s3.create_bucket(Bucket=bucket_name)
    print("S3 Bucket Created Successfully")
except Exception as e:
    print('S3 error', e)

S3 Bucket Created Successfully


In [4]:
#set an output path where the trained model will be saved
prefix='xgboost-as-a-built-in-algo'
output_path='s3://{}/{}/output'.format(bucket_name,prefix)
print(output_path)

s3://flightfareprediction/xgboost-as-a-built-in-algo/output


# Load the Data

In [5]:

import pandas as pd
model_data = pd.read_csv('./Flight_processed.csv',index_col=0)


# Train Test Split of the data

In [6]:

import numpy as np
train_data, test_data = np.split(model_data.sample(frac=1, random_state=1729), [int(0.7 * len(model_data))])
print(train_data.shape, test_data.shape)

(7477, 29) (3205, 29)


# Mapping train And Test Data in S3

In [7]:
#Saving Train and Test Data into Buckets
#Dependent feature has to be in the first column
import os
pd.concat([train_data['Price'],train_data.drop(['Price'],axis=1)],
         axis=1).to_csv('train.csv',index=False,header=False)

boto3.Session().resource('s3').Bucket(bucket_name).Object(os.path.join(prefix, 'train/train.csv')).upload_file('train.csv')

#create path for training folder
s3_input_train = sagemaker.TrainingInput(s3_data='s3://{}/{}/train'.format(bucket_name, prefix), content_type='csv')




In [8]:
#saving Test Data into buckets
pd.concat([test_data['Price'],test_data.drop(['Price'],axis=1)],
         axis=1).to_csv('test.csv',index=False,header=False)

boto3.Session().resource('s3').Bucket(bucket_name).Object(os.path.join(prefix,'test/test.csv')).upload_file('test.csv')

s3_input_test=sagemaker.TrainingInput(s3_data='s3://{}/{}/test'.format(bucket_name, prefix), content_type='csv')

# Building and training Models Xgboost- Inbuilt algorithm

In [9]:
# this line automatically looks for the XGBoost image URI and builds an XGBoost container.
# specify the repo_version depending on your preference.
container = get_image_uri(boto3.Session().region_name,
                          'xgboost', 
                          repo_version='1.0-1')

The method get_image_uri has been renamed in sagemaker>=2.
See: https://sagemaker.readthedocs.io/en/stable/v2.html for details.


In [10]:
# construct a SageMaker estimator that calls the xgboost-container
estimator = sagemaker.estimator.Estimator(image_uri=container, 
                                          role=sagemaker.get_execution_role(),
                                          train_instance_count=1, 
                                          train_instance_type='ml.m5.2xlarge', 
                                          train_volume_size=5, # 5 GB 
                                          output_path=output_path,
                                          train_use_spot_instances=True,
                                          train_max_run=300,
                                          train_max_wait=600)

train_instance_count has been renamed in sagemaker>=2.
See: https://sagemaker.readthedocs.io/en/stable/v2.html for details.
train_instance_type has been renamed in sagemaker>=2.
See: https://sagemaker.readthedocs.io/en/stable/v2.html for details.
train_max_run has been renamed in sagemaker>=2.
See: https://sagemaker.readthedocs.io/en/stable/v2.html for details.
train_use_spot_instances has been renamed in sagemaker>=2.
See: https://sagemaker.readthedocs.io/en/stable/v2.html for details.
train_max_wait has been renamed in sagemaker>=2.
See: https://sagemaker.readthedocs.io/en/stable/v2.html for details.
train_volume_size has been renamed in sagemaker>=2.
See: https://sagemaker.readthedocs.io/en/stable/v2.html for details.


In [11]:
#set hyperparameter
estimator.set_hyperparameters(max_depth=5,
                              eta=0.2,
                              gamma=4,
                              min_child_weight=6,
                              subsample=0.8,
                              silent=0,
                              objective='reg:linear',
                             num_round=50)

In [12]:
#fit the model to train data
estimator.fit({'train': s3_input_train})


2022-02-23 11:33:07 Starting - Starting the training job...
2022-02-23 11:33:30 Starting - Launching requested ML instancesProfilerReport-1645615987: InProgress
......
2022-02-23 11:34:37 Starting - Preparing the instances for training......
2022-02-23 11:35:35 Downloading - Downloading input data
2022-02-23 11:35:35 Training - Downloading the training image.....INFO:sagemaker-containers:Imported framework sagemaker_xgboost_container.training
INFO:sagemaker-containers:Failed to parse hyperparameter objective value reg:linear to Json.
Returning the value itself
INFO:sagemaker-containers:No GPUs detected (normal if no gpus installed)
INFO:sagemaker_xgboost_container.training:Running XGBoost Sagemaker in algorithm mode
INFO:root:Determined delimiter of CSV input is ','
INFO:root:Determined delimiter of CSV input is ','
[11:36:12] 7477x28 matrix with 209356 entries loaded from /opt/ml/input/data/train?format=csv&label_column=0&delimiter=,
INFO:root:Single node training.
INFO:root:Train mat

# Deploy Machine Learning Model as Endpoints

In [13]:
xgb_predictor = estimator.deploy(initial_instance_count=1,instance_type='ml.m4.xlarge')


------!

# Making Predictions of the data


In [15]:
from sagemaker.predictor import csv_serializer, json_deserializer
xgb_predictor.serializer = csv_serializer
xgb_predictor.deserializer = json_deserializer




In [16]:
modelData1 = np.array(train_data.drop(['Price'],axis=1).values).astype('float32')
target1 = np.array(train_data['Price']).astype('float32')

In [18]:
#predicting the Price for the first row
result = xgb_predictor.predict(modelData1[0])
print(result)

The csv_serializer has been renamed in sagemaker>=2.
See: https://sagemaker.readthedocs.io/en/stable/v2.html for details.
The json_deserializer has been renamed in sagemaker>=2.
See: https://sagemaker.readthedocs.io/en/stable/v2.html for details.


3230.130859375


In [19]:
#Actual Price
target1[0]

2754.0

In [ ]:
testdata =  np.array(test_data.drop(['Price'],axis=1).values).astype('int')

predictions = []

for data in testdata:
    result = xgb_predictor.predict(data)


In [25]:
predictions = np.array(predictions)


In [26]:
test_data['predicted_price'] = predictions.astype(int)


In [31]:
test_data.head(10)[[ 'Price', 'predicted_price']]


,Price,predicted_price
2515,3419,4401
8966,10368,12556
2728,12373,13622
10381,14178,11760
1286,8977,7588
9901,9345,8852
9305,3943,4348
9405,8026,6839
967,5126,5432
7225,12557,10182


# Deleting Endpoint

In [34]:
sagemaker.Session().delete_endpoint(xgb_predictor.endpoint)
bucket_to_delete = boto3.resource('s3').Bucket(bucket_name)
bucket_to_delete.objects.all().delete()

The endpoint attribute has been renamed in sagemaker>=2.
See: https://sagemaker.readthedocs.io/en/stable/v2.html for details.


[{'ResponseMetadata': {'RequestId': '8H1B6E1TSZ6QYR84',
   'HostId': 'WsRK9a7kB6/W0QOktwG8/ijuZTG2nCY8hWsaVt8CauypmHlAkTe7BLMyMQI2ShJBC5LCMuf65B8=',
   'HTTPStatusCode': 200,
   'HTTPHeaders': {'x-amz-id-2': 'WsRK9a7kB6/W0QOktwG8/ijuZTG2nCY8hWsaVt8CauypmHlAkTe7BLMyMQI2ShJBC5LCMuf65B8=',
    'x-amz-request-id': '8H1B6E1TSZ6QYR84',
    'date': 'Wed, 23 Feb 2022 19:12:02 GMT',
    'content-type': 'application/xml',
    'transfer-encoding': 'chunked',
    'server': 'AmazonS3',
    'connection': 'close'},
   'RetryAttempts': 0},
  'Deleted': [{'Key': 'xgboost-as-a-built-in-algo/output/sagemaker-xgboost-2022-02-23-11-33-07-179/rule-output/ProfilerReport-1645615987/profiler-output/profiler-reports/OverallFrameworkMetrics.json'},
   {'Key': 'xgboost-as-a-built-in-algo/output/sagemaker-xgboost-2022-02-23-11-33-07-179/rule-output/ProfilerReport-1645615987/profiler-output/profiler-reports/LowGPUUtilization.json'},
   {'Key': 'xgboost-as-a-built-in-algo/output/sagemaker-xgboost-2022-02-23-11-33-07